In [ ]:
import os
import math
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# -------------------------
# User-changeable settings
# -------------------------
dataset_name = "ADNI1GO234"  # used in output filenames
multi_visit_csv = f"./{dataset_name}/filtered/multi_visit_data.csv"   # change if needed
single_visit_csv = f"./{dataset_name}/filtered/single_visit_data.csv" # change if needed
output_dir = f"./{dataset_name}/splits"  # all trial CSVs + params saved here
n_trials = 20
test_frac = 0.20   # fraction of subjects -> test
val_frac_of_train = 0.10  # fraction of TRAIN to use as validation
id_col = "RID"
dx_col = "DXGrp"
# Default covariates - try to detect them in the CSV; fallback behavior below
preferred_cov_cols = ["AGE", "GENDER", "EDUC"]
# Heuristic: phenotype columns start at first column that begins with 'CTX_'
phenotype_prefix = "CTX_"
# -------------------------

os.makedirs(output_dir, exist_ok=True)

In [ ]:
def get_subject_label(dx_values):
    s = set(np.asarray(dx_values).astype(int))
    if s == {1}:
        return "CN"
    if s == {4}:
        return "AD"
    if 4 in s and (2 in s or 3 in s):
        return "pMCI"
    if (2 in s or 3 in s) and not (1 in s or 4 in s):
        return "sMCI"
    return "CNany"

def detect_columns(df):
    # phenotype columns detection
    phen_cols = df.columns[-68:].tolist()
    # covariate detection
    have_pref = [c for c in preferred_cov_cols if c in df.columns]
    cov_cols = preferred_cov_cols.copy()
    return phen_cols, cov_cols

def prepare_subject_table(df):
    """
    Returns DataFrame with one row per subject:
      RID, subject_label, n_records, list_of_rows_idx (optional)
    """
    subj_groups = df.groupby(id_col)
    rows = []
    for rid, g in subj_groups:
        label = get_subject_label(g[dx_col].values)
        rows.append({"RID": rid, "subject_label": label, "n_records": len(g)})
    return pd.DataFrame(rows)

def stratified_subject_split(subj_df, seed):
    """
    Return three lists: train_rids, val_rids, test_rids
    Splits are stratified by subject_label.
    """
    # first split into train_val and test
    rids = subj_df["RID"].values
    labels = subj_df["subject_label"].values

    trainval_rids, test_rids, trainval_lbls, test_lbls = train_test_split(
        rids, labels, test_size=test_frac, random_state=seed, stratify=labels
    )

    # now split trainval into train and val using val_frac_of_train fraction of trainval as val
    if len(trainval_rids) == 0:
        return [], [], test_rids.tolist()

    val_size = val_frac_of_train
    train_rids, val_rids, train_lbls, val_lbls = train_test_split(
        trainval_rids, trainval_lbls, test_size=val_size, random_state=seed + 1, stratify=trainval_lbls
    )

    return train_rids.tolist(), val_rids.tolist(), test_rids.tolist()


In [ ]:
# -------------------------
# Preadjust functions
# -------------------------
def adjust_phenotype_HC_return_params(pheno, covs, dx, dx_code):
    n_sbj, n_res = pheno.shape
    n_cov = covs.shape[1]
    adj_pheno = np.full(pheno.shape, np.nan)
    betas = np.full((n_res, n_cov + 1), np.nan)  # intercept + covs
    means = np.full((n_res, n_cov), np.nan)

    if dx_code < 0:
        # use all subjects
        # find rows where all covariates are non-missing
        non_miss_covs_idx = np.arange(n_sbj)
        for i in range(n_cov):
            non_miss_covs_idx = np.intersect1d(non_miss_covs_idx, np.where(~np.isnan(covs[:, i]))[0])

        covs_new = np.column_stack((np.ones(n_sbj), covs))  # intercept
        for i_res in range(n_res):
            resp = pheno[:, i_res]
            non_miss_pheno = np.where(~np.isnan(resp))[0]
            non_miss = np.intersect1d(non_miss_pheno, non_miss_covs_idx)
            if len(non_miss) > 0:
                beta = np.linalg.lstsq(covs_new[non_miss, :], resp[non_miss], rcond=None)[0]
                betas[i_res, :] = beta.ravel()
                B = beta[1:].ravel()
                M = np.mean(covs_new[non_miss, :], axis=0)[1:]
                means[i_res, :] = M
                # apply adjustment
                for j_idx in non_miss:
                    adj_pheno[j_idx, i_res] = pheno[j_idx, i_res] - np.sum(B * (covs[j_idx, :] - M))
    else:
        # fit on subset dx==dx_code
        sbj_idx = np.where(dx == dx_code)[0]
        if len(sbj_idx) == 0:
            # nothing to fit - return NaNs
            return adj_pheno, betas, means
        covs3 = covs[sbj_idx, :]
        pheno3 = pheno[sbj_idx, :]

        non_miss_covs_idx = np.arange(len(sbj_idx))
        for i in range(n_cov):
            non_miss_covs_idx = np.intersect1d(non_miss_covs_idx, np.where(~np.isnan(covs3[:, i]))[0])

        covs_new = np.column_stack((np.ones(len(sbj_idx)), covs3))

        # For applying to all subjects, find rows where covariates available
        non_miss_covs_all = np.arange(n_sbj)
        for i in range(n_cov):
            non_miss_covs_all = np.intersect1d(non_miss_covs_all, np.where(~np.isnan(covs[:, i]))[0])

        for i_res in range(n_res):
            resp = pheno3[:, i_res]
            non_miss_pheno = np.where(~np.isnan(resp))[0]
            non_miss = np.intersect1d(non_miss_pheno, non_miss_covs_idx)

            resp_all = pheno[:, i_res]
            non_miss_pheno_all = np.where(~np.isnan(resp_all))[0]
            non_miss_all = np.intersect1d(non_miss_pheno_all, non_miss_covs_all)

            if len(non_miss) > 0:
                # Regression using least squares
                beta = np.linalg.lstsq(covs_new[non_miss, :], resp[non_miss], rcond=None)[0]
                betas[i_res, :] = beta.ravel()
                B = beta[1:].ravel()
                M = np.mean(covs_new[non_miss, :], axis=0)[1:]
                means[i_res, :] = M
                # apply adjustment for all non_miss_all
                for j_idx in non_miss_all:
                    adj_pheno[j_idx, i_res] = pheno[j_idx, i_res] - np.sum(B * (covs[j_idx, :] - M))

    return adj_pheno, betas, means

def apply_preadjust_params(pheno, covs, betas, means):
    n_sbj, n_res = pheno.shape
    adj = np.full(pheno.shape, np.nan)
    n_cov = covs.shape[1]
    for i_res in range(n_res):
        if np.all(np.isnan(betas[i_res, :])):
            continue  # no params
        B = betas[i_res, 1:].ravel()
        M = means[i_res, :].ravel()
        # find subjects with non-missing phenotype AND non-missing covs
        non_miss_pheno_all = np.where(~np.isnan(pheno[:, i_res]))[0]
        non_miss_covs_all = np.arange(n_sbj)
        for k in range(n_cov):
            non_miss_covs_all = np.intersect1d(non_miss_covs_all, np.where(~np.isnan(covs[:, k]))[0])
        non_miss_all = np.intersect1d(non_miss_pheno_all, non_miss_covs_all)
        for j_idx in non_miss_all:
            adj[j_idx, i_res] = pheno[j_idx, i_res] - np.sum(B * (covs[j_idx, :] - M))
    return adj


# Processing

In [ ]:
def run_processing():
    print("Loading CSV:", multi_visit_csv)
    df = pd.read_csv(multi_visit_csv)
    print("Loading CSV:", single_visit_csv)
    df_single = pd.read_csv(single_visit_csv)
    if id_col not in df.columns or dx_col not in df.columns:
        raise RuntimeError(f"Expecting columns {id_col} and {dx_col} in input CSV")

    phen_cols, covs_selected = detect_columns(df)
    print("Detected phenotype columns:", len(phen_cols))
    print("Using covariate columns:", covs_selected)

    # Build subject-level table and labels
    subj_df = prepare_subject_table(df)
    print("Unique subjects:", len(subj_df))

    # For each trial:
    for t in range(n_trials):
        seed = 1000 + t
        print(f"\n--- Trial {t} (seed={seed}) ---")
        train_rids, val_rids, test_rids = stratified_subject_split(subj_df, seed)

        # get splits as dataframes (rows for all visits of subjects)
        train_df = df[df[id_col].isin(train_rids)].reset_index(drop=True)
        val_df = df[df[id_col].isin(val_rids)].reset_index(drop=True)
        test_df = df[df[id_col].isin(test_rids)].reset_index(drop=True)

        # Save CSVs for this trial
        trial_path = os.path.join(output_dir, f"trial{t}")
        os.makedirs(trial_path, exist_ok=True)
        train_csv = f"{trial_path}/train.csv"
        val_csv = f"{trial_path}/val.csv"
        test_csv = f"{trial_path}/test.csv"

        train_df.to_csv(train_csv, index=False)
        val_df.to_csv(val_csv, index=False)
        test_df.to_csv(test_csv, index=False)
        print(f"Saved {train_csv}, {val_csv}, {test_csv}")

        # ------------ preadjust on TRAIN ------------
        covs_train = train_df[covs_selected].astype(float).values
        covs_val = val_df[covs_selected].astype(float).values
        covs_test = test_df[covs_selected].astype(float).values
        covs_single = df_single[covs_selected].astype(float).values
        pheno_train = train_df[phen_cols].astype(float).values
        pheno_val = val_df[phen_cols].astype(float).values
        pheno_test = test_df[phen_cols].astype(float).values
        pheno_single = df_single[phen_cols].astype(float).values
        dx_train = train_df[dx_col].astype(int).values
        dx_all_for_apply = pd.concat([train_df, val_df, test_df], ignore_index=True)[dx_col].astype(int).values

        dx_code = 1
        adj_train, betas, means = adjust_phenotype_HC_return_params(pheno_train, covs_train, dx_train, dx_code)

        # Save parameters for this trial
        params_file = f"{trial_path}/preadj_params.npz"
        np.savez(params_file, betas=betas, means=means, cov_names=np.array(covs_selected), phen_names=np.array(phen_cols))
        print(f"Saved preadjust parameters to {params_file}")

        # Save adjusted TRAIN data (merge back into DataFrame)
        adj_train_df = pd.DataFrame(adj_train, columns=phen_cols)
        # keep identifier columns from train_df and replace phenotype columns with adjusted values
        train_df_adj = train_df.copy()
        train_df_adj[phen_cols] = adj_train_df[phen_cols]
        train_preadj_csv = f"{trial_path}/preadj_train.csv"
        train_df_adj.to_csv(train_preadj_csv, index=False)
        print(f"Saved adjusted TRAIN: {train_preadj_csv}")
        
        # Apply parameters to VAL
        adj_val = apply_preadjust_params(pheno_val, covs_val, betas, means)
        val_df_adj = val_df.copy()
        val_df_adj[phen_cols] = pd.DataFrame(adj_val, columns=phen_cols)
        val_preadj_csv = f"{trial_path}/preadj_val.csv"
        val_df_adj.to_csv(val_preadj_csv, index=False)
        print(f"Saved adjusted VAL: {val_preadj_csv}")

        # Apply parameters to TEST
        adj_test = apply_preadjust_params(pheno_test, covs_test, betas, means)
        test_df_adj = test_df.copy()
        test_df_adj[phen_cols] = pd.DataFrame(adj_test, columns=phen_cols)
        test_preadj_csv = f"{trial_path}/preadj_test.csv"
        test_df_adj.to_csv(test_preadj_csv, index=False)
        print(f"Saved adjusted TEST: {test_preadj_csv}")
        
        # Apply parameters to single visit data
        adj_single = apply_preadjust_params(pheno_single, covs_single, betas, means)
        single_df_adj = df_single.copy()
        single_df_adj[phen_cols] = pd.DataFrame(adj_single, columns=phen_cols)
        single_preadj_csv = f"{trial_path}/preadj_single.csv"
        single_df_adj.to_csv(single_preadj_csv, index=False)
        print(f"Saved adjusted SINGLE: {single_preadj_csv}")

    print("\nAll trials finished. Outputs in:", os.path.abspath(output_dir))

In [ ]:
run_processing()